In [1]:
%load_ext autoreload
%autoreload 2

import equinox as eqx
import jax
import jax.random as jr
import jax.numpy as jnp
import numpy as np
import optax

from trainax import (
    BestModelSaver,
    Callback,
    EpochLogger,
    JaxLoader,
    LossMetricTracker,
    PbarHandler,
    StepOutput,
    Trainer,
    ValStepOutput
)

jax.config.update("jax_num_cpu_devices", 4)

In [2]:
SEED = 42
NTRAIN = 1000
NVAL = 200
M = 20
BATCH_SIZE = 200
rng = np.random.RandomState(SEED)

In [26]:
def loss_fun(model: eqx.Module, data: dict):
    yhat = jax.vmap(model)(data["x"])
    return jnp.mean((yhat - data["y"]) ** 2), yhat

def make_step(model: eqx.Module, data: dict) -> StepOutput:
    (loss, yhat), grads = eqx.filter_value_and_grad(loss_fun, has_aux=True)(model, data)
    return StepOutput(loss=loss, y=data["y"], yhat=yhat, gradients=grads)

def val_step(model: eqx.Module, data: dict) -> ValStepOutput:
    inference_model = eqx.nn.inference_mode(model)
    loss, yhat = loss_fun(inference_model, data)
    return ValStepOutput(loss=loss, y=data["y"], yhat=yhat)

In [4]:
train_data = {
    "x": rng.randn(NTRAIN, M),
    "y": rng.randint(0, 2, NTRAIN)
}
val_data = {
    "x": rng.randn(NVAL, M),
    "y": rng.randint(0, 2, NVAL)
}

In [5]:
train_loader = JaxLoader(train_data, batch_size=BATCH_SIZE)
val_loader = JaxLoader(val_data, batch_size=BATCH_SIZE)

In [6]:
model = eqx.nn.MLP(
    in_size=M,
    out_size=1,
    width_size=10,
    depth=5,
    key=jr.PRNGKey(SEED)
)

In [59]:
# TODO: add additional callbacks
callbacks = [
    EpochLogger(),
    PbarHandler(),
]

In [50]:
# TODO: test manual sharding
data_sharding = [0, 1, 2, 3]

In [89]:
# TODO: implement trainer
optim = optax.adam(learning_rate=1e-3)
trainer = Trainer(
    n_epochs=20,
    val_every=2,
    callbacks=callbacks,
    data_sharding=data_sharding,
)

In [91]:
train_out = trainer.train(
    model=model,
    optim=optim,
    trainloader=train_loader,
    valloader=val_loader,
    train_step=make_step,
    val_step=val_step,
)

Output()

In [92]:
train_out

TrainOutput(model=MLP(
  layers=(
    Linear(
      weight=f32[10,20],
      bias=f32[10],
      in_features=20,
      out_features=10,
      use_bias=True
    ),
    Linear(
      weight=f32[10,10],
      bias=f32[10],
      in_features=10,
      out_features=10,
      use_bias=True
    ),
    Linear(
      weight=f32[10,10],
      bias=f32[10],
      in_features=10,
      out_features=10,
      use_bias=True
    ),
    Linear(
      weight=f32[10,10],
      bias=f32[10],
      in_features=10,
      out_features=10,
      use_bias=True
    ),
    Linear(
      weight=f32[10,10],
      bias=f32[10],
      in_features=10,
      out_features=10,
      use_bias=True
    ),
    Linear(
      weight=f32[1,10],
      bias=f32[1],
      in_features=10,
      out_features=1,
      use_bias=True
    )
  ),
  activation=<PjitFunction of <function relu at 0x10cdfe980>>,
  final_activation=<function <lambda>>,
  use_bias=True,
  use_final_bias=True,
  in_size=20,
  out_size=1,
  width_size=10,
  d

In [ ]:
from pydantic import ConfigDict
from pydantic.dataclasses import dataclass

@dataclass(config=ConfigDict(arbitrary_types_allowed=True))
class TestOutput:
    y: jnp.ndarray

jax.tree_util.register_dataclass(TestOutput)

def mult(x: jnp.ndarray, val: float) -> TestOutput:
    return TestOutput(y=x * val)